# Computer Use Through the API

Let the model complete a simple form in an isolated demonstration browser, while our Python application executes actions and returns screenshots.

[Open in Google Colab](https://colab.research.google.com/github/BrightPool/udemy-prompt-engineering-course/blob/main/openai_features_and_functionality/computer_use.ipynb) · [Official guide](https://developers.openai.com/api/docs/guides/tools-computer-use)

**Connection to the earlier lesson:** Computer Use can also be integrated into your own application through the API. The model chooses actions; **your app supplies the browser and executes them**.

**Filming path (about 4 minutes):** pre-run setup, show the form, explain the screenshot/action loop, run the task, and inspect the filled form. The helper cell is provided so beginners can focus on the API request.

We use the native `computer` tool because it makes the loop visible. Current docs also describe a code-execution approach, recommended for GPT-6 Astra. You need API billing and access to a model supporting the computer tool. The demo uses only a local synthetic page and blocks external requests.

In [ ]:
%pip install openai playwright ipython --upgrade --quiet

In [ ]:
RUN_DEMO = True
MAX_TURNS = 12  # Bounds the number of model responses and associated API usage.

In [ ]:
import os
import sys
import platform
import subprocess
import asyncio
import base64
from getpass import getpass
from pathlib import Path
from openai import OpenAI
from playwright.async_api import async_playwright
from IPython.display import display, Markdown, FileLink, Image

if not isinstance(RUN_DEMO, bool):
    raise TypeError("RUN_DEMO must be True or False, not text.")
if RUN_DEMO and not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(timeout=180, max_retries=1) if RUN_DEMO else None
MODEL = os.getenv("OPENAI_COMPUTER_USE_MODEL", "gpt-6.1-sol")
print("Model:", MODEL)

In [ ]:
# Run once before filming. Colab/Linux needs Chromium and its system dependencies.
if not isinstance(RUN_DEMO, bool):
    raise TypeError("RUN_DEMO must be True or False, not text.")
install = [sys.executable, "-m", "playwright", "install"]
if platform.system() == "Linux":
    install.append("--with-deps")
if RUN_DEMO:
    subprocess.run(install + ["chromium"], check=True)

## 1. Our controlled test environment

This browser has a fresh, unauthenticated context. The page submits **no data** and has no network connections. We ask the model to fill two fields and press a preview button. This avoids real websites, logins, purchases, or access to your own desktop.

In [ ]:
DEMO_HTML = """
<!doctype html><html><head><title>Course contact form</title>
<style>
body {font:22px Arial; padding:55px; background:#f5f6fa; color:#242424}
main {background:white; padding:35px; width:560px}
label {display:block; margin:20px 0 8px}
input {font:22px Arial; padding:12px; width:490px}
button {font:22px Arial; padding:14px 22px; margin-top:25px}
#preview {margin-top:25px; color:#00695c}
</style></head><body><main><h1>Course contact form</h1>
<p>Demo only. Nothing is sent or saved online.</p>
<label for="name">Name</label><input id="name" autocomplete="off">
<label for="email">Email</label><input id="email" autocomplete="off">
<button id="preview-button" onclick="
document.querySelector('#preview').textContent =
'Preview: ' + document.querySelector('#name').value + ' | ' + document.querySelector('#email').value;
">Preview</button><p id="preview"></p></main></body></html>
"""
pw = browser = context = page = None
if RUN_DEMO:
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=True)
    context = await browser.new_context(viewport={"width": 1024, "height": 768})
    await context.route("**/*", lambda route: route.abort())
    page = await context.new_page()
    await page.set_content(DEMO_HTML)
    display(Image(data=await page.screenshot()))

## 2. How the application translates actions

This helper supports the mouse and keyboard actions used by the native computer tool. It **does not run model-generated Python or shell commands**. Unsupported actions and safety-check requests stop execution. For this local demo, navigation is blocked and a turn limit bounds the loop.

For a real integration, add domain restrictions, meaningful approval controls, and an isolated environment appropriate to the task. The helper below is a teaching example, not a complete production permission system.

In [ ]:
KEYS = {"ENTER": "Enter", "TAB": "Tab", "ESC": "Escape", "ESCAPE": "Escape",
        "BACKSPACE": "Backspace", "DELETE": "Delete", "SPACE": "Space",
        "CTRL": "Control", "CONTROL": "Control", "ALT": "Alt", "SHIFT": "Shift",
        "CMD": "Meta", "META": "Meta", "ARROWUP": "ArrowUp", "ARROWDOWN": "ArrowDown",
        "ARROWLEFT": "ArrowLeft", "ARROWRIGHT": "ArrowRight"}

async def execute_actions(page, actions):
    for action in actions:
        kind = action["type"]
        if kind == "screenshot":
            continue
        elif kind == "click":
            button = {"wheel": "middle"}.get(action.get("button", "left"), action.get("button", "left"))
            await page.mouse.click(action["x"], action["y"], button=button)
        elif kind == "double_click":
            await page.mouse.dblclick(action["x"], action["y"])
        elif kind == "move":
            await page.mouse.move(action["x"], action["y"])
        elif kind == "type":
            await page.keyboard.insert_text(action["text"])
        elif kind == "keypress":
            keys = [KEYS.get(key.upper(), key) for key in action["keys"]]
            await page.keyboard.press("+".join(keys))
        elif kind == "scroll":
            await page.mouse.move(action["x"], action["y"])
            await page.mouse.wheel(action["scroll_x"], action["scroll_y"])
        elif kind == "drag":
            points = action["path"]
            if not points:
                raise ValueError("An empty drag path is not supported.")
            await page.mouse.move(points[0]["x"], points[0]["y"])
            await page.mouse.down()
            try:
                for point in points[1:]:
                    await page.mouse.move(point["x"], point["y"])
            finally:
                await page.mouse.up()
        elif kind == "wait":
            await asyncio.sleep(0.5)
        else:
            raise ValueError(f"Unsupported action: {kind}")
        if page.url != "about:blank":
            raise RuntimeError("Navigation away from the local demo page is not allowed.")

async def computer_result(page, call):
    if getattr(call, "pending_safety_checks", None):
        raise RuntimeError("The model requested a safety check. Stop for human review.")
    actions = [action.model_dump() for action in call.actions]
    print("Requested actions:", [action["type"] for action in actions])
    await execute_actions(page, actions)
    screenshot = base64.b64encode(await page.screenshot()).decode("ascii")
    return {"type": "computer_call_output", "call_id": call.call_id,
            "output": {"type": "computer_screenshot",
                       "image_url": f"data:image/png;base64,{screenshot}", "detail": "original"}}

## 3. Send a task, execute actions, send back the new screen

We send a **screenshot of the current page** with our initial request. Every returned `computer_call` is handled in order. Its result carries the same `call_id`; `previous_response_id` continues the conversation while the browser remains open.

Notice that the API does not click anything by itself. Our helper performs each allowed action and sends the updated screen back to the model.

In [ ]:
if RUN_DEMO:
    try:
        screenshot = base64.b64encode(await page.screenshot()).decode("ascii")
        next_input = [{"role": "user", "content": [
            {"type": "input_text", "text": (
                "Use the computer tool to fill this local demo form. "
                "Set Name to Alex Learner and Email to alex@example.com. "
                "Click Preview, check that the preview contains both values, then stop. "
                "Do not navigate or open other sites.")},
            {"type": "input_image", "image_url": f"data:image/png;base64,{screenshot}", "detail": "original"},
        ]}]
        previous_response_id = None
        for turn in range(MAX_TURNS):
            response = await asyncio.to_thread(
                client.responses.create, model=MODEL, tools=[{"type": "computer"}],
                input=next_input, previous_response_id=previous_response_id,
                max_output_tokens=1500,
            )
            if response.status != "completed":
                raise RuntimeError(f"Response stopped with status {response.status}.")
            calls = [item for item in response.output if item.type == "computer_call"]
            if not calls:
                unexpected = [item.type for item in response.output if item.type not in {"message", "reasoning"}]
                if unexpected:
                    raise RuntimeError(f"Unexpected tool output: {unexpected}")
                display(Markdown(response.output_text))
                break
            next_input = [await computer_result(page, call) for call in calls]
            previous_response_id = response.id
        else:
            raise RuntimeError("Turn limit reached. Inspect the result before starting a new run.")

        # Verify the real browser state, rather than relying on the model's answer.
        assert await page.locator("#name").input_value() == "Alex Learner"
        assert await page.locator("#email").input_value() == "alex@example.com"
        assert await page.locator("#preview").inner_text() == "Preview: Alex Learner | alex@example.com"
        display(Image(data=await page.screenshot()))
        print("Verified: the fields and preview contain the requested values.")
    finally:
        if browser is not None:
            await browser.close()
        if pw is not None:
            await pw.stop()

## Try it yourself

Change the requested name and email **and the verification values**, then rerun from the environment cell. Compare the actions the model chooses. Keep the exercise on this synthetic page.

## Summary

- Computer Use is available through the API as well as the chat experiences shown earlier.
- Your application supplies a persistent browser or desktop and executes the requested actions.
- Return screenshots with matching call IDs, repeat the loop, and verify the real UI state.
- Use an isolated environment, narrow permissions, human approval where needed, and execution limits.

**Troubleshooting:** rerun the browser-install cell if Chromium is missing; use the async Playwright API with top-level `await` in Jupyter/Colab; choose a computer-capable model if the tool is rejected. If a cell fails before the final cell, run `await browser.close()` and `await pw.stop()` to release the demo environment.

Sources: [Computer Use](https://developers.openai.com/api/docs/guides/tools-computer-use), [Integration recipes](https://developers.openai.com/api/docs/guides/tools-computer-use-integration), [Playwright async API](https://playwright.dev/python/docs/library#usage).